# SurgeFlow live monitor

The monitor runs every available offering on its cadence. Offerings 03 and 09 use member research routes (beta).

US cohorts are enabled by default. Set `SURGEFLOW_COHORT_MARKETS` to a comma-separated market list to enable other cohorts.

Cohort and layoff panels refresh at most once every six hours. **Feed unavailable** is an availability state, not an error.

The default Hotlist market is `us`. Fixed weekday exchange hours gate that panel; holidays are not handled.

Each panel refreshes on a schedule; each panel shows its source date. Changed rows receive a highlight (or a `changed` mark when jinja2 is not installed).

Research data only, not investment advice. Free access grants no redistribution rights. **Do not republish the data you fetch.**

The 1,500 request budget counts only this running monitor. A restart starts it again from zero and spends about 22 requests on the first full refresh.

The key's 2,000 requests per day are shared by every process using it. The budget is capped once at startup from the reported remaining count.

The default projection is 1,234 scheduled requests plus one account check. All four cohort markets give 1,246 plus one account check.

Fixture mode runs one full refresh and stops; `SURGEFLOW_MONITOR_MAX_CYCLES` sets another bound. Run the Stop cell to end a live monitor.

In [ ]:
import asyncio
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetched_at, fixture_mode, get_api_key, make_client
from surgeflow.monitor import Monitor, RequestBudget, default_panels, seed_budget_from_rate_limit
client, catalogue = make_client()
api_key = get_api_key()
cohort_markets = tuple(
    market.strip().lower()
    for market in os.environ.get('SURGEFLOW_COHORT_MARKETS', 'us').split(',')
    if market.strip()
)
print('Enabled cohort markets:', ', '.join(cohort_markets) or 'none')
FIXTURE_NOW = datetime(2026, 11, 23, 15, 0, tzinfo=timezone.utc)
def monitor_now():
    return FIXTURE_NOW if fixture_mode() else datetime.now(timezone.utc)
budget = RequestBudget(daily_limit=1500)
budget.consume(1, monitor_now())
try:
    me = client.member_me(api_key)
    print('Plan:', me.get('plan'), '; usage:', me.get('usage'))
    remaining = seed_budget_from_rate_limit(budget, me.get('rate_limit'))
    print('Key daily requests remaining across all processes:', remaining if remaining is not None else 'not supplied',
          '; local budget used:', budget.used, '/', budget.daily_limit)
except Exception as exc:
    print('Account check error:', type(exc).__name__, str(exc), '; fetched at:', fetched_at())
monitor = Monitor(
    default_panels(client, api_key, cohort_markets=cohort_markets),
    budget=budget,
    now=monitor_now,
)
projected = monitor.print_projection()
print('Projected total including one startup account check:', projected + 1)

In [ ]:
import os
STOP = False
# 0 means run until stopped; fixture mode defaults to one full refresh.
MAX_CYCLES = int(os.environ.get('SURGEFLOW_MONITOR_MAX_CYCLES') or (1 if fixture_mode() else 0))
async def refresh_loop():
    cycles = 0
    while not STOP:
        try:
            monitor.run_due()
            monitor.render()
        except Exception as exc:
            print('Error in refresh cycle:', type(exc).__name__, str(exc), '; at:', fetched_at())
        cycles += 1
        if MAX_CYCLES and cycles >= MAX_CYCLES:
            break
        await asyncio.sleep(1)
def report_task_exception(task):
    if not task.cancelled() and task.exception() is not None:
        exc = task.exception()
        print('Error: monitor task stopped:', type(exc).__name__, str(exc), '; at:', fetched_at())
monitor_task = asyncio.create_task(refresh_loop())
monitor_task.add_done_callback(report_task_exception)
if MAX_CYCLES:
    await monitor_task
else:
    await asyncio.sleep(0)

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print('Optional ipywidgets is not installed. Use the Stop cell.')
else:
    stop_button = widgets.Button(description='Stop monitor')
    def stop_button_clicked(_):
        global STOP
        STOP = True
    stop_button.on_click(stop_button_clicked)
    display(stop_button)

## Stop
Run the next cell to stop the refresh loop.

In [ ]:
STOP = True
if not monitor_task.done():
    monitor_task.cancel()
print('Stop requested. Requests used locally:', monitor.budget.used, '/', monitor.budget.daily_limit)